# Qwen抑制指示 × 意味的距離：4概念パイロット実験

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryouy/GraduationThesis/blob/main/1112_qwen_suppression_pilot/qwen_suppression_pilot_4concepts.ipynb)

[研究プロトコル](https://github.com/ryouy/GraduationThesis/blob/main/docs/main_experiment_protocol.md) · [固定済み64 Cue](https://github.com/ryouy/GraduationThesis/tree/main/1112_cue_semantic_distance/final_stimuli/20261112_frozen_ai_audit_v1) · [過去のProbe独立評価](https://github.com/ryouy/GraduationThesis/blob/main/1112_qwen_probe_independent_confirmation/results/20261112_newfacts_16targets_v1/analysis_summary.md)

**目的**：固定した4概念×16 Cue×4条件とCueなし対照（**272入力**）について、Qwenで**1,360応答（5反復）**を生成し、出力へのX言及・固定X/Y Probeの判別値・層別Logit Lensを測る。

**区別**：
- これは**パイロット**。CueはAI監査済みだが人間による独立評定が未完了。距離を意図的に変えた文章には意味・具体性の違いも含まれる。
- **ROC-AUC ≠ Xの活性量**。過去のX/Y学習データだけでProbeを凍結し、同じX/Y・同じ層の**指示条件差**を調べる。
- **Logit Lens ≠ 中間層での実際の出力**。層1〜23は最終RMSNormで投影、24層はすでに最終Norm済み。文字列が複数トークンならteacher-forcingでトークン列全体を評価する。
- **意味的言及のAPI判定はオプション**。Colab Secret `OPENAI_API_KEY` がなければ未評価と記録し、明示的言及は常に算出する。API判定には別途料金がかかる。

**実行**：Colab GPU（T4可）を選択。`GITHUB_RESULTS_TOKEN`（リポジトリのContents読み書き）をSecretsで有効化。実行は上から順番に。途中終了後は**同じノートブックを再実行**するとGitHubの検証済みチェックポイントを復元。設計変更時は `RUN_ID` を更新し既存結果を上書きしない。

In [ ]:
%pip -q install "transformers==4.57.6" "accelerate>=1,<2" "huggingface-hub>=0.30,<1" "scikit-learn==1.6.1" pandas numpy requests tqdm matplotlib scipy tabulate

## 0. セットアップ・原データの凍結

今回の64 CueだけをGitHubから読み込む。変更追跡版の原本を使用し、未採用Cueや生成結果に基づく再選定はしない。元のWikipediaから作った学習特徴は別のGitHubチェックポイントから**読み取り専用**で取り出す。

In [ ]:
import os, io, re, json, math, hashlib, zipfile, base64, time, unicodedata, random
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from google.colab import userdata

REPO='ryouy/GraduationThesis'
ROOT='1112_qwen_suppression_pilot'
RUN_ID='20261112_qwen_suppression_pilot_v1'
DEST=f'{ROOT}/results/{RUN_ID}'
CUE_SOURCE='1112_cue_semantic_distance/final_stimuli/20261112_frozen_ai_audit_v1'
PROBE_SOURCE='1111_qwen_probe_100concepts_v2/results/20261112_qwen100_richer_dedup_v2'
CONFIRM_SOURCE='1112_qwen_probe_independent_confirmation/results/20261112_newfacts_16targets_v1'
MODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'
MODEL_REVISION='7ae557604adf67be50417f59c2c2f167def9a775'
LAYERS=[22,23,24]
CONCEPT_Y={'boat':'bus','bus':'helicopter','bicycle':'submarine','motorcycle':'helicopter'}
CONDITIONS=['Base','Think X','Avoid X','Think Y']
BANDS=['near','medium','far','unrelated','none']
SEEDS=[101,211,307,401,503]
TEMP=0.7; TOP_P=0.9; MAX_NEW_TOKENS=128
CHUNK_INPUTS=16
WORK=Path('/content/qwen_suppression_pilot')/RUN_ID
CACHE=WORK/'checkpoints';CACHE.mkdir(parents=True,exist_ok=True)
OUT=WORK/'results';OUT.mkdir(parents=True,exist_ok=True)
np.random.seed(42);random.seed(42);torch.manual_seed(42)
try: GITHUB_TOKEN=userdata.get('GITHUB_RESULTS_TOKEN')
except Exception: GITHUB_TOKEN=None
assert GITHUB_TOKEN, 'Please enable GITHUB_RESULTS_TOKEN in Colab Secrets (repository Contents read/write).'
G=requests.Session();G.headers.update({'Authorization':'Bearer '+GITHUB_TOKEN,'Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'})
API='https://api.github.com/repos/'+REPO+'/contents/'

def remote_meta(path):
    response=G.get(API+path,timeout=90)
    if response.status_code==404:return None
    response.raise_for_status();return response.json()

def read_remote(path):
    info=remote_meta(path)
    if info is None:raise FileNotFoundError(path)
    response=G.get(info['download_url'],timeout=240)
    response.raise_for_status();return response.content

def maybe_remote(path):
    info=remote_meta(path)
    if info is None:return None
    response=G.get(info['download_url'],timeout=240)
    response.raise_for_status();return response.content

def publish_immutable(path,data,message):
    if isinstance(data,str):data=data.encode('utf-8')
    git_sha=hashlib.sha1(b'blob '+str(len(data)).encode()+b'\0'+data).hexdigest()
    info=remote_meta(path)
    if info is not None:
        if info['sha']!=git_sha:raise RuntimeError('Different immutable artifact exists: '+path+'. Use a NEW RUN_ID.')
        print('Verified existing:',path);return
    payload={'branch':'main','message':message,'content':base64.b64encode(data).decode('ascii')}
    response=G.put(API+path,json=payload,timeout=240)
    response.raise_for_status();print('Saved:',path)

def sha256(x):return hashlib.sha256(x).hexdigest()
def dump_json(obj):return (json.dumps(obj,ensure_ascii=False,indent=2,sort_keys=True)+'\n').encode()
def save_df(df,name):df.to_csv(OUT/name,index=False)

def jsonl_bytes(rows):return ('\n'.join(json.dumps(x,ensure_ascii=False,sort_keys=True) for x in rows)+'\n').encode()
def read_jsonl(blob):return [json.loads(line) for line in blob.decode('utf-8').splitlines() if line.strip()]
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT SELECTED')

## 1. 固定Cue・X/Yペア・プロンプトを検証する

BaseではX名を入力しない。**CueなしのBase**は4概念で入力本文が一致するため、概念ごとに独立した4種類の刺激として解釈しない。全条件とも同じchat templateのuserメッセージとして入力する。

In [ ]:
cue_raw=read_remote(CUE_SOURCE+'/cue_final_FROZEN_AI_AUDITED.csv')
cue=pd.read_csv(io.BytesIO(cue_raw),keep_default_na=False)
cue_manifest=json.loads(read_remote(CUE_SOURCE+'/curation_manifest.json'))
assert cue_manifest['status']=='FROZEN_FOR_PILOT_AI_REVIEWED_HUMAN_PENDING'
assert cue_manifest['final_count']==64 and not cue_manifest['human_verified']
assert len(cue)==64 and cue.cue_id.nunique()==64
assert set(cue.target_x)==set(CONCEPT_Y)
assert all(len(g)==16 for _,g in cue.groupby('target_x'))
assert cue.apply(lambda r: CONCEPT_Y[r.target_x]==r.comparator_y,axis=1).all()
assert cue.apply(lambda r: sum((cue.target_x==r.target_x)&(cue.distance_band==r.distance_band))==4,axis=1).all()
assert set(cue.distance_band)==set(BANDS)-{'none'}
assert not cue.cue_text.duplicated().any()
assert not cue.cue_text.str.contains(r'\b(?:boat|bus|bicycle|motorcycle|helicopter|submarine)\b',case=False,regex=True).any(), 'Explicit label present in cue'

# Treat all cues as distinct trial stimuli; none is added separately to avoid changing the frozen 64.
INSTRUCTIONS={
 'Base': 'Read the clue and briefly name the thing it brings to mind.',
 'Think X': 'Think about {X}. Read the clue and briefly name the thing it brings to mind.',
 'Avoid X': 'Do not think about {X}. Read the clue and briefly name the thing it brings to mind.',
 'Think Y': 'Think about {Y} instead of {X}. Read the clue and briefly name the thing it brings to mind.'
}

def construct_prompt(x,y,condition,cue_text):
    inst=INSTRUCTIONS[condition].replace('{X}',x).replace('{Y}',y)
    tail='Clue: '+cue_text+'\nAnswer with one short phrase.' if cue_text else 'No clue is provided.\nAnswer with one short phrase.'
    return inst+'\n'+tail

rows=[]
for x in sorted(CONCEPT_Y):
    y=CONCEPT_Y[x]
    subset=cue[cue.target_x==x].sort_values('cue_id')
    for r in subset.to_dict('records'):
        for cond in CONDITIONS:
            rows.append({'input_id':x+'__'+r['cue_id']+'__'+cond.replace(' ','_'),
                'target_x':x,'comparator_y':y,'cue_id':r['cue_id'],'cue_text':r['cue_text'],
                'distance_band':r['distance_band'],'distance_x':float(r['distance_x']),
                'distance_y':float(r['distance_y']),'delta_x_minus_y':float(r['delta_x_minus_y']),
                'condition':cond,'is_no_cue':False,
                'user_prompt':construct_prompt(x,y,cond,r['cue_text'])})
    for cond in CONDITIONS:
        rows.append({'input_id':x+'__NO_CUE__'+cond.replace(' ','_'),
            'target_x':x,'comparator_y':y,'cue_id':'NO_CUE','cue_text':'',
            'distance_band':'none','distance_x':np.nan,'distance_y':np.nan,'delta_x_minus_y':np.nan,
            'condition':cond,'is_no_cue':True,
            'user_prompt':construct_prompt(x,y,cond,'')})
PROMPTS=pd.DataFrame(rows).sort_values(['target_x','cue_id','condition']).reset_index(drop=True)
assert len(PROMPTS)==272 and PROMPTS.input_id.is_unique
assert (PROMPTS.condition.value_counts()==68).all()
assert PROMPTS.is_no_cue.sum()==16
PROMPT_BYTES=PROMPTS.to_csv(index=False).encode()
PROMPT_HASH=sha256(PROMPT_BYTES); CUE_HASH=sha256(cue_raw)
save_df(PROMPTS,'prompts.csv')
save_df(cue,'cue_selected.csv')
print('Cue SHA256:',CUE_HASH,'Prompt SHA256:',PROMPT_HASH)
print(PROMPTS[['target_x','distance_band','condition']].groupby(['target_x','distance_band']).size().unstack())

## 2. 以前の学習Hidden Stateを復元してProbeを凍結

前回の`train`だけを復元。**評価データは分類器に入れず**、本パイロットの指示条件を使った再学習も行わない。過去の16文中8学習文、scaler + LogisticRegression(C=0.1)を同じ実装で再構成する。

In [ ]:
old_manifest=json.loads(read_remote(PROBE_SOURCE+'/manifest.json'))
old_raw=read_remote(PROBE_SOURCE+'/stimulus_design.csv')
assert old_manifest['status']=='COMPLETED_EXPLORATORY_SCREEN'
assert old_manifest['revision']==MODEL_REVISION and old_manifest['model']==MODEL_ID
assert old_manifest['layers']==LAYERS and old_manifest['position']=='assistant_start'
assert sha256(old_raw)==old_manifest['prompt_hash']
old_stim=pd.read_csv(io.BytesIO(old_raw),keep_default_na=False)
assert len(old_stim)==1104
REF=pd.read_csv(io.BytesIO(read_remote(CONFIRM_SOURCE+'/frozen_pair_confirmation.csv')))
for x,y in CONCEPT_Y.items():
    p='__'.join(sorted((x,y)))
    row=REF[REF.pair==p]
    assert len(row)==1 and bool(row.iloc[0].confirmation_pass_fdr), 'Frozen contrast did not pass independent sensitivity check: '+p
print('Frozen comparisons validated:',[(x,CONCEPT_Y[x]) for x in sorted(CONCEPT_Y)])

PREV_BATCH=64
N_OLD=len(old_stim)
N_OLD_BATCH=math.ceil(N_OLD/PREV_BATCH)
N_OLD_GROUP=math.ceil(N_OLD_BATCH/4)
OLD_FEATURES=[]
for group in tqdm(range(N_OLD_GROUP),desc='Read previous immutable training checkpoints'):
    blob=read_remote(f'{PROBE_SOURCE}/checkpoints/chunk_{group:03d}.zip')
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        cfg=json.loads(archive.read('config.json'))
        assert cfg['prompt_hash']==old_manifest['prompt_hash'] and cfg['layers']==LAYERS
        for b in range(group*4,min((group+1)*4,N_OLD_BATCH)):
            with np.load(io.BytesIO(archive.read(f'batch_{b:03d}.npz')),allow_pickle=False) as z:
                assert str(z['prompt_hash'].item())==old_manifest['prompt_hash']
                assert int(z['start'])==b*PREV_BATCH
                assert int(z['stop'])==min(N_OLD,(b+1)*PREV_BATCH)
                v=z['features'].copy()
                assert v.shape==(min(N_OLD,(b+1)*PREV_BATCH)-b*PREV_BATCH,3,896)
                OLD_FEATURES.append(v)
OLD_FEATURES=np.concatenate(OLD_FEATURES,axis=0)
assert OLD_FEATURES.shape==(N_OLD,3,896)
OLD_INDEX={(r.concept,r.split,int(r.family)):i for i,r in enumerate(old_stim.itertuples())}
assert len(OLD_INDEX)==len(old_stim)
def old_train(name,layer_index):
    return np.stack([OLD_FEATURES[OLD_INDEX[name,'train',k],layer_index] for k in range(8)])
PROBES={};probe_records=[]
for x,y in CONCEPT_Y.items():
    for li,layer in enumerate(LAYERS):
        X=np.concatenate([old_train(x,li),old_train(y,li)],axis=0)
        labels=np.r_[np.ones(8,dtype=int),np.zeros(8,dtype=int)]
        scaler=StandardScaler().fit(X)
        clf=LogisticRegression(C=0.1,solver='liblinear',max_iter=1000,random_state=42).fit(scaler.transform(X),labels)
        PROBES[x,layer]=(scaler,clf)
        checksum=sha256(np.concatenate([scaler.mean_,scaler.scale_,clf.coef_.ravel(),clf.intercept_]).tobytes())
        refrow=REF[REF.pair=='__'.join(sorted((x,y)))].iloc[0]
        probe_records.append({'target_x':x,'comparator_y':y,'layer':layer,
          'train_examples_per_label':8,'positive_label':x,'negative_label':y,
          'probe_param_sha256':checksum,'prior_fresh_L24_auc':float(refrow.L24_auc),
          'prior_confirmation_pass_fdr':bool(refrow.confirmation_pass_fdr)})
PROBE_INFO=pd.DataFrame(probe_records)
save_df(PROBE_INFO,'probe_fit_validation.csv')
assert len(PROBES)==12
print(PROBE_INFO[['target_x','comparator_y','layer','prior_fresh_L24_auc']].to_string(index=False))
del OLD_FEATURES

## 3. 実験設定をGitHubに先に固定する

同一RUN_IDで再実行した場合はファイル内容のSHAを確認。異なるデータに変更して上書きすることはできない。

In [ ]:
CONFIG={
 'run_id':RUN_ID,'experiment_phase':'pilot','model':MODEL_ID,'model_revision':MODEL_REVISION,
 'probe_training_source':PROBE_SOURCE,'probe_original_prompt_hash':old_manifest['prompt_hash'],
 'confirmation_source':CONFIRM_SOURCE,'stimulus_source':CUE_SOURCE,
 'cue_sha256':CUE_HASH,'prompts_sha256':PROMPT_HASH,
 'fixed_pairs':CONCEPT_Y,'conditions':CONDITIONS,'n_inputs':len(PROMPTS),
 'n_repeats':len(SEEDS),'seeds':SEEDS,'temperature':TEMP,'top_p':TOP_P,
 'max_new_tokens':MAX_NEW_TOKENS,'chunk_size_inputs':CHUNK_INPUTS,
 'primary_hidden_position':'last_input_token_in_assistant_prefix',
 'probe_layers':LAYERS,'logit_lens_layers':list(range(1,25)),
 'probe_spec':'StandardScaler + LogisticRegression(C=0.1,solver=liblinear,random_state=42)',
 'explicit_dictionary_version':'pilot-v1','ai_cue_review_only':True,
 'semantic_grader':'optional, separately versioned; not part of immutable core result'
}
publish_immutable(DEST+'/config.json',dump_json(CONFIG),'Freeze Qwen suppression pilot config')
for name in ['prompts.csv','cue_selected.csv','probe_fit_validation.csv']:
    publish_immutable(DEST+'/'+name,(OUT/name).read_bytes(),'Freeze Qwen suppression pilot '+name)

## 4. GPU上で生成・Hidden State・Logit Lensを取得（チェックポイント対応）

- Hidden State：生成**前**、assistant開始直前の最後の入力トークンについて272個を1回ずつ測る。
- Probe：**同じX/Y Probe内の判別スコア**（22/23/24層）。
- Logit Lens：1〜24層の**候補ラベル列全体**のteacher-forced log probability。前の語を固定して予測し、途中層を語彙空間へ投影する補助指標。最終層のlogits一致をセル内でテストする。
- 生成：入力ごとにseed5個で5回答。**同じ入力の5反復でHidden Stateは共通**。

In [ ]:
assert torch.cuda.is_available(),'Select GPU / T4 runtime in Colab.'
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=MODEL_REVISION,use_fast=True)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=MODEL_REVISION,torch_dtype=torch.float32,attn_implementation='eager').to('cuda').eval()
assert model.config.num_hidden_layers==24 and model.config.hidden_size==896
assert hasattr(model.model,'norm') and hasattr(model,'lm_head')
EOT_IDS=[i for i in [tokenizer.eos_token_id,tokenizer.convert_tokens_to_ids('<|im_end|>')] if isinstance(i,int) and i>=0]
EOT_IDS=sorted(set(EOT_IDS))

def encode_prompt(text):
    return tokenizer.apply_chat_template([{'role':'user','content':text}],tokenize=True,
       add_generation_prompt=True,return_tensors='pt').to('cuda')

def hidden_to_lens_logits(hidden,layer):
    # Qwen2 final hidden state (layer 24) has already passed final RMSNorm.
    vec=hidden if layer==24 else model.model.norm(hidden)
    return model.lm_head(vec).float()

@torch.inference_mode()
def lens_for_label(input_ids,original_forward,label):
    tokens=tokenizer.encode(label,add_special_tokens=False)
    assert 1<=len(tokens)<=8,(label,tokens)
    if len(tokens)==1:
        states=original_forward.hidden_states
    else:
        append=torch.tensor([tokens[:-1]],device='cuda',dtype=input_ids.dtype)
        extended=torch.cat((input_ids,append),dim=1)
        states=model(input_ids=extended,output_hidden_states=True,use_cache=False).hidden_states
    out=[]
    for layer in range(1,25):
        # Last n positions = last original prefix position followed by n-1 appended token positions.
        h=states[layer][0,-len(tokens):,:]
        logits=hidden_to_lens_logits(h,layer)
        logp=F.log_softmax(logits,dim=-1)
        chosen=logp[torch.arange(len(tokens),device='cuda'),torch.tensor(tokens,device='cuda')]
        out.append({'layer':layer,'sequence_logprob':float(chosen.sum().item()),
                    'mean_token_logprob':float(chosen.mean().item()),
                    'first_token_logit':float(logits[0,tokens[0]].item())})
    return tokens,out

@torch.inference_mode()
def execute_input(r):
    inp=encode_prompt(r.user_prompt)
    fwd=model(input_ids=inp,output_hidden_states=True,use_cache=False)
    # Correctness guard: layer-24 hidden is already normalized and should match the native logits.
    exact=model.lm_head(fwd.hidden_states[-1][0,-1,:]).float()
    native=fwd.logits[0,-1,:].float()
    err=float((exact-native).abs().max().item())
    assert err<1e-4, f'Final-layer Logit Lens mismatch: {err}'
    feat=np.stack([fwd.hidden_states[L][0,-1,:].float().cpu().numpy() for L in LAYERS]).astype(np.float32)
    probe=[]
    for idx,layer in enumerate(LAYERS):
        ss,clf=PROBES[r.target_x,layer]
        score=float(clf.decision_function(ss.transform(feat[idx:idx+1]))[0])
        probe.append({'input_id':r.input_id,'target_x':r.target_x,'comparator_y':r.comparator_y,
           'layer':layer,'x_vs_y_decision':score})
    lenses=[]
    for name,label in [('X',r.target_x),('Y',r.comparator_y)]:
        ids,quant=lens_for_label(inp,fwd,label)
        for ent in quant:
            lenses.append({'input_id':r.input_id,'candidate_role':name,'candidate_text':label,
               'token_ids':' '.join(map(str,ids)),'token_count':len(ids),**ent})
    del fwd
    if torch.cuda.is_available():torch.cuda.empty_cache()
    generations=[]
    for repeat,seed in enumerate(SEEDS):
        # transformers GenerationMixin does not reliably accept generator= as a kwarg.
        # Seed PyTorch immediately before sampling; document GPU nondeterminism separately.
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        sampled=model.generate(input_ids=inp,do_sample=True,temperature=TEMP,top_p=TOP_P,
            max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            eos_token_id=EOT_IDS,use_cache=True)
        output_ids=sampled[0,inp.shape[1]:].tolist()
        output_text=tokenizer.decode(output_ids,skip_special_tokens=True).strip()
        generations.append({'response_id':r.input_id+f'__repeat{repeat}',
            'input_id':r.input_id,'target_x':r.target_x,'comparator_y':r.comparator_y,
            'cue_id':r.cue_id,'distance_band':r.distance_band,'distance_x':None if pd.isna(r.distance_x) else float(r.distance_x),
            'condition':r.condition,'repeat_index':repeat,'seed':seed,'response':output_text,
            'output_token_count':len(output_ids),'truncated_at_limit':len(output_ids)>=MAX_NEW_TOKENS and (not output_ids or output_ids[-1] not in EOT_IDS),
            'empty_response':not bool(output_text)})
    return feat,probe,lenses,generations,err

# An explicit single-input GPU smoke check is integrated into the first chunk; no data are discarded.
print('Model ready. Hidden state and native logits alignment is checked for every input.')

## 5. Immutable checkpoints: 入力16件ずつ保存・中断時に復旧

モデルの生成が完了するまで最終結果とは扱わない。チェックポイントにはプロンプトハッシュ・入力範囲・特徴ベクトル・Probe/Logit Lens・5回生成の生データを含める。

In [ ]:
N=len(PROMPTS);N_CHUNKS=math.ceil(N/CHUNK_INPUTS)
def remote_chunk(g):return f'{DEST}/checkpoints/chunk_{g:03d}.zip'
def local_chunk(g):return CACHE/f'chunk_{g:03d}.zip'

def verify_chunk(g,blob):
    begin=g*CHUNK_INPUTS;end=min(N,(g+1)*CHUNK_INPUTS)
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        cfg=json.loads(z.read('config.json'))
        assert cfg==CONFIG,'Changed config: use NEW RUN_ID'
        metadata=json.loads(z.read('meta.json'))
        assert metadata['prompt_hash']==PROMPT_HASH
        assert metadata['begin']==begin and metadata['end']==end
        with np.load(io.BytesIO(z.read('hidden_features.npz')),allow_pickle=False) as f:
            data=f['features']
            assert data.shape==(end-begin,len(LAYERS),896) and np.isfinite(data).all()
        generations=read_jsonl(z.read('generations.jsonl'))
        probes=read_jsonl(z.read('probe_scores.jsonl'))
        lenses=read_jsonl(z.read('logit_lens.jsonl'))
        assert len(generations)==(end-begin)*len(SEEDS)
        assert len(probes)==(end-begin)*len(LAYERS)
        assert len(lenses)==(end-begin)*24*2
        assert {a['input_id'] for a in generations}==set(PROMPTS.input_id.iloc[begin:end])
        assert len({a['response_id'] for a in generations})==len(generations)
    return True

def recover_chunk(g):
    remote=maybe_remote(remote_chunk(g))
    if remote is not None:
        verify_chunk(g,remote)
        local_chunk(g).write_bytes(remote)
        print('Recovered',g,'from GitHub')
        return True
    if local_chunk(g).exists():
        verify_chunk(g,local_chunk(g).read_bytes())
        return True
    return False

def execute_chunk(g):
    begin=g*CHUNK_INPUTS;end=min(N,(g+1)*CHUNK_INPUTS)
    features=[];probes=[];lenses=[];generations=[];errors=[]
    for r in tqdm(PROMPTS.iloc[begin:end].itertuples(index=False),total=end-begin,desc=f'Chunk {g:03d}'):
        f,p,l,gen,err=execute_input(r)
        features.append(f);probes.extend(p);lenses.extend(l);generations.extend(gen);errors.append(err)
    assert len(features)==end-begin
    f=io.BytesIO()
    np.savez_compressed(f,features=np.stack(features),prompt_hash=np.array(PROMPT_HASH))
    metadata={'begin':begin,'end':end,'prompt_hash':PROMPT_HASH,'final_layer_max_logit_error':max(errors)}
    buf=io.BytesIO()
    with zipfile.ZipFile(buf,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.writestr('config.json',dump_json(CONFIG))
        z.writestr('meta.json',dump_json(metadata))
        z.writestr('hidden_features.npz',f.getvalue())
        z.writestr('probe_scores.jsonl',jsonl_bytes(probes))
        z.writestr('logit_lens.jsonl',jsonl_bytes(lenses))
        z.writestr('generations.jsonl',jsonl_bytes(generations))
    raw=buf.getvalue();verify_chunk(g,raw)
    local_chunk(g).write_bytes(raw)
    # Local presence does NOT mean remote checkpoint exists. Publish after a new chunk.
    # If a remote checkpoint appeared during execution, validate and prefer it without overwriting.
    remote=maybe_remote(remote_chunk(g))
    if remote is None:
        publish_immutable(remote_chunk(g),raw,f'Qwen suppression pilot checkpoint {g:03d}')
    else:
        verify_chunk(g,remote)
        local_chunk(g).write_bytes(remote)
    return True

for g in range(N_CHUNKS):recover_chunk(g)
for g in range(N_CHUNKS):
    if local_chunk(g).exists():continue
    execute_chunk(g)
for g in range(N_CHUNKS):assert recover_chunk(g)
print('All',N_CHUNKS,'chunks complete and GitHub backed.')

## 6. 特徴ベクトルと生の生成結果をまとめる

明示的言及は**単語境界を含む事前固定辞書**で判定する。否定文の中に単語がある場合も語の言及としては`True`、意味的な「Xそのものを答えたか」とは区別する。

In [ ]:
features=[];generations=[];probes=[];lenses=[];lens_errors=[]
for g in range(N_CHUNKS):
    verify_chunk(g,local_chunk(g).read_bytes())
    with zipfile.ZipFile(local_chunk(g),'r') as z:
        with np.load(io.BytesIO(z.read('hidden_features.npz')),allow_pickle=False) as f:
            features.append(f['features'].copy())
        generations+=read_jsonl(z.read('generations.jsonl'))
        probes+=read_jsonl(z.read('probe_scores.jsonl'))
        lenses+=read_jsonl(z.read('logit_lens.jsonl'))
        lens_errors.append(json.loads(z.read('meta.json'))['final_layer_max_logit_error'])
FEATURES=np.concatenate(features)
assert FEATURES.shape==(272,3,896)
GEN=pd.DataFrame(generations); PROBE=pd.DataFrame(probes); LENS=pd.DataFrame(lenses)
assert len(GEN)==1360 and len(PROBE)==272*3 and len(LENS)==272*24*2
assert GEN.response_id.is_unique
assert PROBE[['input_id','layer']].drop_duplicates().shape[0]==len(PROBE)
assert LENS[['input_id','layer','candidate_role']].drop_duplicates().shape[0]==len(LENS)
print('Max native vs reconstructed final-layer logits mismatch:',max(lens_errors))

# High-precision dictionary. Broader paraphrase/negation handling is deliberately deferred to API grading.
ALIASES={
 'boat':['boat','boats','watercraft','dinghy','dinghies','rowboat','rowboats','motorboat','motorboats'],
 'bus':['bus','buses','coach','coaches','shuttle bus','shuttle buses'],
 'bicycle':['bicycle','bicycles','bike','bikes','pushbike','pushbikes'],
 'motorcycle':['motorcycle','motorcycles','motorbike','motorbikes','motorized two wheeler','motorized two wheelers']
}
PATTERNS={x:re.compile(r'(?<![A-Za-z])(?:'+'|'.join(re.escape(v) for v in sorted(aliases,key=len,reverse=True))+r')(?![A-Za-z])',re.IGNORECASE) for x,aliases in ALIASES.items()}
def explicit_mention(x,text):
    normalized=unicodedata.normalize('NFKC',str(text))
    return bool(PATTERNS[x].search(normalized))
GEN['explicit_mention']=[explicit_mention(row.target_x,row.response) for row in GEN.itertuples()]
GEN['format_issue']=GEN.empty_response | GEN.truncated_at_limit | (GEN.output_token_count>64)
GEN['semantic_label']='not_run'
save_df(GEN,'generations.csv');save_df(PROBE,'probe_scores.csv');save_df(LENS,'logit_lens.csv')
summary=GEN.groupby(['target_x','distance_band','condition'],dropna=False).agg(
 n=('explicit_mention','size'),explicit_rate=('explicit_mention','mean'),
 format_issue_rate=('format_issue','mean'),mean_tokens=('output_token_count','mean')).reset_index()
save_df(summary,'explicit_mention_by_condition_band.csv')
print(summary.to_string(index=False))

## 7. 主たる対照比較・距離×条件の可視化

関連Cueの距離は連続値。説明図では帯を使用する。**独立観測単位はCue**であり、生成5回を独立したCueとして扱わない。概念ごとに条件差を算出する。無関係CueとCueなしは距離回帰に混ぜない。

In [ ]:
# Per unique target/cue/condition (5 repeats -> one empirical rate)
CUE_RATES=GEN.groupby(['target_x','cue_id','distance_band','condition'],as_index=False).agg(
    explicit_rate=('explicit_mention','mean'),n_repeats=('explicit_mention','size'),
    truncated_rate=('truncated_at_limit','mean'))
CUE_RATES=CUE_RATES.merge(PROMPTS[['target_x','cue_id','distance_band','distance_x']].drop_duplicates(),
  on=['target_x','cue_id','distance_band'],how='left',validate='many_to_one')
WIDE=CUE_RATES.pivot(index=['target_x','cue_id','distance_band'],columns='condition',values='explicit_rate').reset_index()
WIDE=WIDE.merge(PROMPTS[['target_x','cue_id','distance_band','distance_x']].drop_duplicates(),
 on=['target_x','cue_id','distance_band'],validate='one_to_one')
assert len(WIDE)==68
WIDE['avoid_minus_thinkx']=WIDE['Avoid X']-WIDE['Think X']
WIDE['avoid_minus_base']=WIDE['Avoid X']-WIDE['Base']
WIDE['thinkx_minus_base']=WIDE['Think X']-WIDE['Base']
save_df(WIDE,'cue_level_explicit_effects.csv')

P_SCORE=PROBE.merge(PROMPTS[['input_id','target_x','cue_id','distance_band','distance_x','condition']],
  on=['input_id','target_x'],how='left',validate='many_to_one')
PW=P_SCORE.pivot(index=['target_x','cue_id','distance_band','layer'],
    columns='condition',values='x_vs_y_decision').reset_index()
PW=PW.merge(PROMPTS[['target_x','cue_id','distance_band','distance_x']].drop_duplicates(),
 on=['target_x','cue_id','distance_band'],validate='many_to_one')
assert len(PW)==204
PW['avoid_minus_thinkx']=PW['Avoid X']-PW['Think X']
PW['avoid_minus_base']=PW['Avoid X']-PW['Base']
save_df(PW,'cue_level_probe_effects.csv')

LENS_PAIR=LENS.pivot(index=['input_id','layer'],columns='candidate_role',values='sequence_logprob').reset_index()
LENS_PAIR['x_minus_y_sequence_logprob']=LENS_PAIR['X']-LENS_PAIR['Y']
LENS_PAIR=LENS_PAIR.merge(PROMPTS[['input_id','target_x','cue_id','condition','distance_band','distance_x']],on='input_id',validate='many_to_one')
save_df(LENS_PAIR,'logit_lens_pair_margins.csv')

RELATED=WIDE[WIDE.distance_band.isin(['near','medium','far'])].copy()
RELATED_P=PW[(PW.layer==24)&PW.distance_band.isin(['near','medium','far'])].copy()
for name,frame,measure in [
    ('generation',RELATED,'avoid_minus_thinkx'),('probe_L24',RELATED_P,'avoid_minus_thinkx')]:
    print(name, 'per-X paired effects (Avoid-ThinkX):')
    print(frame.groupby('target_x')[measure].agg(['size','mean','std']).to_string())

fig,axes=plt.subplots(1,2,figsize=(12,4.3))
order=['near','medium','far']
for condition in CONDITIONS:
    arr=[CUE_RATES[(CUE_RATES.distance_band==b)&(CUE_RATES.condition==condition)].explicit_rate.mean() for b in order]
    axes[0].plot(order,arr,marker='o',label=condition)
axes[0].set(ylim=(-.03,1.03),ylabel='Explicit X mention rate (cue-level mean)',xlabel='Distance band',title='Generation: distance × instruction')
axes[0].legend()
for target in sorted(CONCEPT_Y):
    arr=[RELATED_P[(RELATED_P.target_x==target)&(RELATED_P.distance_band==b)].avoid_minus_thinkx.mean() for b in order]
    axes[1].plot(order,arr,marker='o',label=target)
axes[1].axhline(0,color='gray',lw=1)
axes[1].set(xlabel='Distance band',ylabel='Avoid − Think X (within-pair Probe score)',title='Probe L24: paired difference, by concept')
axes[1].legend()
fig.tight_layout()
fig.savefig(OUT/'pilot_overview.png',dpi=170,bbox_inches='tight')
plt.show()

fig,axes=plt.subplots(1,2,figsize=(12,4))
for target in sorted(CONCEPT_Y):
    seq=RELATED[RELATED.target_x==target].sort_values('distance_x')
    axes[0].scatter(seq.distance_x,seq.avoid_minus_thinkx,label=target,s=30)
    seqp=RELATED_P[RELATED_P.target_x==target].sort_values('distance_x')
    axes[1].scatter(seqp.distance_x,seqp.avoid_minus_thinkx,label=target,s=30)
axes[0].set(xlabel='Cosine distance to X',ylabel='Avoid - Think X (explicit mention)',title='Per-cue output difference')
axes[1].set(xlabel='Cosine distance to X',ylabel='Avoid - Think X (Probe L24)',title='Per-cue within-probe difference')
for ax in axes:ax.axhline(0,color='gray',lw=1);ax.legend()
fig.tight_layout()
fig.savefig(OUT/'cue_level_condition_effects.png',dpi=170,bbox_inches='tight')
plt.show()

# Compare conditions *within each X/Y pair*; do not pool unequal label tokenizations or cross-probe scales.
LW=LENS_PAIR.pivot(index=['target_x','cue_id','distance_band','layer'],
   columns='condition',values='x_minus_y_sequence_logprob').reset_index()
LW['avoid_minus_thinkx']=LW['Avoid X']-LW['Think X']
LW['avoid_minus_base']=LW['Avoid X']-LW['Base']
save_df(LW,'logit_lens_cue_level_effects.csv')
fig,ax=plt.subplots(figsize=(10,5))
for target in sorted(CONCEPT_Y):
    t=LW[(LW.target_x==target)&(LW.distance_band.isin(['near','medium','far']))]
    g=t.groupby('layer').avoid_minus_thinkx.mean()
    ax.plot(g.index,g.values,label=target)
ax.set(xlabel='Transformer layer (not time)',ylabel='Avoid − Think X, within-pair log P(X) − log P(Y)',title='Logit Lens paired condition difference')
ax.axhline(0,color='gray',lw=1);ax.legend()
fig.tight_layout();fig.savefig(OUT/'logit_lens_layerwise.png',dpi=170,bbox_inches='tight')
plt.show()

## 8. 結果レポートと再現用ファイルをGitHubへ保存

**API判定がなくても完了できるコア測定**。生成・辞書言及・Probe・Logit Lensのみの結果として固定し、API判定を行う場合は別の名前空間に保存する。

In [ ]:
MANIFEST={**CONFIG,'status':'COMPLETED_CORE_PILOT_SEMANTIC_GRADING_SEPARATE',
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,
 'checkpoint_count':N_CHUNKS,'n_outputs':len(GEN),'n_probe_scores':len(PROBE),
 'n_lens_rows':len(LENS),'max_final_layer_logit_error':max(lens_errors),
 'human_cue_review':'PENDING',
 'limitations':['4 concepts are all vehicles','Only AI-audited cue quality; no independent human semantic judgments',
 'Cue distance is not isolated from meaning and specificity',
 'ROC-AUC from earlier data is not a measure of concept activation',
 'Binary generation mentions from 5 correlated repeats per cue are not independent stimuli',
 'Instructions explicitly mention X except Base, and any instruction effect may include lexical priming',
 'Multitoken logit lens uses teacher-forced fixed label strings, not free generation',
 'Semantic grading is optional and separately stored']}
# Preserve prior completion metadata when resuming the same immutable run.
prior_manifest_blob=maybe_remote(DEST+'/manifest.json')
if prior_manifest_blob is not None:
    prior_manifest=json.loads(prior_manifest_blob)
    assert all(prior_manifest[k]==MANIFEST[k] for k in CONFIG), 'Published manifest/config differ; choose a new run ID.'
    MANIFEST=prior_manifest
(OUT/'manifest.json').write_bytes(dump_json(MANIFEST))

r=summary[summary.distance_band.isin(['near','medium','far'])]
sum_table=r.pivot_table(index=['target_x','distance_band'],columns='condition',values='explicit_rate').round(3).to_markdown()
effect_table=RELATED.groupby('target_x').agg(n_cues=('cue_id','nunique'),
 avoid_minus_thinkx=('avoid_minus_thinkx','mean'),avoid_minus_base=('avoid_minus_base','mean')).round(3).to_markdown()
probe_table=RELATED_P.groupby('target_x').agg(n_cues=('cue_id','nunique'),
 avoid_minus_thinkx=('avoid_minus_thinkx','mean'),avoid_minus_base=('avoid_minus_base','mean')).round(3).to_markdown()
report=f"""# Qwen抑制指示パイロット — コア測定結果

- **状態：** COMPLETED_CORE_PILOT_SEMANTIC_GRADING_SEPARATE
- **モデル：** {MODEL_ID} @ {MODEL_REVISION}
- **入力：** {len(PROMPTS)}（4X×16Cue×4条件＋Cueなし16）
- **生成：** {len(GEN)}（5反復/入力）
- **Cue原本のSHA256：** `{CUE_HASH}`
- **プロンプト設計のSHA256：** `{PROMPT_HASH}`
- **Logit Lens最終層のnative logits一致（最大誤差）：** {max(lens_errors):.8g}
- **明示的X言及：** 辞書判定。否定文中の語も言及と数える。
- **意味的X言及：** このファイルでは**未実施**。APIによる独立の後続判定は別出力に保存。

## 距離帯×指示条件：明示的X言及率

{sum_table}

## Cue単位で平均した指示効果（Avoid X − Think X / Base）

{effect_table}

## 第24層Probeの条件差（各概念内でのみ解釈）

{probe_table}

## 分析上の注意

- 結果は4つの乗り物概念に限る。CueはAI監査後の固定64文であり、独立人手検証は未了。
- 出力はCueごとの5生成をまとめて分析。Cue数は関連Cue48、無関係Cue16であり、1,360件を独立Cue扱いしない。
- Probeの出力はX/Yの**相対判別スコア**。異なる概念で生スコアを平均して「活性化量」とは呼ばない。
- 近い/中/遠Cueは内容や具体性も異なるため、観測された差を意味的距離だけの因果効果とみなさない。
- 無関係CueとCueなしを距離回帰に混ぜない。BaseのCueなしは全Xで同じ文字列。
- [プロトコル](https://github.com/{REPO}/blob/main/docs/main_experiment_protocol.md)の事前検討事項を踏まえた探索的なパイロット。
"""
(OUT/'analysis_summary.md').write_text(report,encoding='utf-8')
files=['manifest.json','analysis_summary.md','config.json','prompts.csv','cue_selected.csv',
 'probe_fit_validation.csv','generations.csv','probe_scores.csv','logit_lens.csv',
 'explicit_mention_by_condition_band.csv','cue_level_explicit_effects.csv',
 'cue_level_probe_effects.csv','logit_lens_pair_margins.csv','logit_lens_cue_level_effects.csv',
 'pilot_overview.png','cue_level_condition_effects.png','logit_lens_layerwise.png']
for name in files:
    publish_immutable(DEST+'/'+name,(OUT/name).read_bytes(),'Publish Qwen suppression pilot '+name)
print('Completed results: https://github.com/'+REPO+'/tree/main/'+DEST)

## 9. 任意：LLM APIによる**意味的言及**判定（別途課金）

Colab Secret `OPENAI_API_KEY` を登録した場合だけ、このセルで1,360応答の判定を実行。明示的言及とは異なり、**Xそのものを指す／同定するか**を評価する。「湖」「車輪」のような関連語だけではyesにしない。曖昧な回答はuncertain。

判定器には**X名と回答文字列だけ**を渡し、Cueや指示条件を見せない。API利用がなければ「未判定」として終了。API評価は数十件ずつのチェックポイントを別フォルダへ保存し、再実行可能にする。モデル名と判定指示は事前固定。注意：API側のモデルは更新・廃止される場合があり、**応答のモデルIDと実行時刻**を保存する。

[OpenAI Responses API](https://platform.openai.com/docs/api-reference/responses/create) · [料金](https://platform.openai.com/pricing)

In [ ]:
try: OPENAI_KEY=userdata.get('OPENAI_API_KEY')
except Exception: OPENAI_KEY=None
GRADE_MODEL='gpt-6-luna'  # Confirm model availability in your own API account before running.
GRADE_RUN_ID='20261112_semantic_grader_gpt6luna_v1'
GRADE_DEST=f'{DEST}/semantic_grading/{GRADE_RUN_ID}'
GRADE_BATCH_SIZE=40
JUDGE_PROMPT="""
You judge whether a short English answer identifies a given target concept.
You will be given ONLY target and answer; you do not know the instructions or clue.
Return yes if the answer asserts or clearly identifies the target itself, including clear paraphrases.
Return no if it names something else, merely associates with the target, or explicitly negates it.
Return uncertain if genuinely ambiguous; do not guess based on context you have not seen.
A word occurrence in a negation is not a semantic yes. A related scene, object, or property is not a yes.
Keep rationale_short under 20 words. Return only the required JSON schema.
"""
GRADE_SCHEMA={'type':'object','properties':{
    'label':{'type':'string','enum':['yes','no','uncertain']},
    'rationale_short':{'type':'string'}},
    'required':['label','rationale_short'],'additionalProperties':False}
GRADE_CONFIG={'grader_model_requested':GRADE_MODEL,'grader_prompt_sha256':sha256(JUDGE_PROMPT.encode()),
    'grader_schema_sha256':sha256(dump_json(GRADE_SCHEMA)),
    'source_run':RUN_ID,'generation_response_sha256':sha256((OUT/'generations.csv').read_bytes()),
    'grade_batch_size':GRADE_BATCH_SIZE,'number_responses':len(GEN)}

if not OPENAI_KEY:
    print('OPENAI_API_KEY not configured. Semantic Mention Rate remains NOT RUN. Core pilot is complete.')
else:
    publish_immutable(GRADE_DEST+'/config.json',dump_json(GRADE_CONFIG),'Freeze semantic grading config')
    GRADE_SESSION=requests.Session();GRADE_SESSION.headers.update({'Authorization':'Bearer '+OPENAI_KEY,'Content-Type':'application/json'})
    def call_semantic_grader(target,response):
        payload={'model':GRADE_MODEL,
          'instructions':JUDGE_PROMPT,
          'input':f'Target: {target}\nAnswer: {response}',
          'text':{'format':{'type':'json_schema','name':'semantic_mention','strict':True,'schema':GRADE_SCHEMA}},
          'max_output_tokens':320}
        for attempt in range(6):
            res=GRADE_SESSION.post('https://api.openai.com/v1/responses',json=payload,timeout=120)
            if res.status_code in [429,500,502,503,504]:
                time.sleep(min(2**attempt,30));continue
            res.raise_for_status()
            obj=res.json();pieces=[]
            for block in obj.get('output',[]):
                if block.get('type')!='message':continue
                for p in block.get('content',[]):
                    if p.get('type')=='output_text':pieces.append(p['text'])
            if not pieces:raise RuntimeError('API response has no output_text; do not score as no: '+str(obj.get('status')))
            data=json.loads(''.join(pieces))
            assert data['label'] in ['yes','no','uncertain']
            return {'semantic_label':data['label'],'rationale_short':data['rationale_short'],
                'grader_model_returned':obj.get('model',''),'grader_response_id':obj.get('id','')}
        raise RuntimeError('Grading API retry limit reached; resume without modifying RUN_ID')

    all_grades=[]
    for b in tqdm(range(math.ceil(len(GEN)/GRADE_BATCH_SIZE)),desc='Semantic grade batches'):
        start=b*GRADE_BATCH_SIZE;stop=min(len(GEN),(b+1)*GRADE_BATCH_SIZE)
        remote=f'{GRADE_DEST}/checkpoints/grades_{b:03d}.jsonl'
        blob=maybe_remote(remote)
        if blob is None:
            graded=[]
            for r in GEN.iloc[start:stop].itertuples(index=False):
                rec=call_semantic_grader(r.target_x,r.response)
                graded.append({'response_id':r.response_id,'target_x':r.target_x,**rec})
            blob=jsonl_bytes(graded)
            publish_immutable(remote,blob,f'Semantic grader batch {b:03d}')
        recovered=read_jsonl(blob)
        assert len(recovered)==stop-start
        assert [row['response_id'] for row in recovered]==GEN.response_id.iloc[start:stop].tolist()
        all_grades+=recovered
    GRADED=pd.DataFrame(all_grades)
    assert len(GRADED)==len(GEN) and GRADED.response_id.is_unique
    joined=GEN.merge(GRADED,on=['response_id','target_x'],how='left',validate='one_to_one',suffixes=('','_graded'))
    assert joined.semantic_label_graded.isin(['yes','no','uncertain']).all()
    joined['semantic_yes']=joined.semantic_label_graded.eq('yes')
    rated=joined.groupby(['target_x','distance_band','condition'],as_index=False).agg(
        count=('semantic_yes','size'),semantic_yes_rate=('semantic_yes','mean'),
        uncertainty_rate=('semantic_label_graded',lambda x:(x=='uncertain').mean()))
    grade_dir=OUT/'semantic_grading';grade_dir.mkdir(exist_ok=True)
    joined.to_csv(grade_dir/'graded_responses.csv',index=False)
    rated.to_csv(grade_dir/'semantic_mention_by_condition_band.csv',index=False)
    summary_text='# Optional semantic grading of Qwen outputs\n\n'+'- Model requested: '+GRADE_MODEL+'\n'+'- Answers graded: '+str(len(GRADED))+'\n'+'- Uncertain count: '+str((GRADED.semantic_label=='uncertain').sum())+'\n\n'+rated.to_markdown(index=False)+'\n\nThis is model-based grading and requires independent manual spot checks.\n'
    (grade_dir/'analysis_semantic.md').write_text(summary_text)
    grade_manifest={**GRADE_CONFIG,'status':'COMPLETED_API_SEMANTIC_GRADING','graded_count':len(GRADED),
        'actual_grader_models':sorted(GRADED.grader_model_returned.unique().tolist()),
        'completed_utc':datetime.now(timezone.utc).isoformat(),
        'human_validation':'PENDING','caveat':'Do not equate related mentions with identity judgments; uncertain excluded from yes count'}
    (grade_dir/'manifest.json').write_bytes(dump_json(grade_manifest))
    for filename in ['graded_responses.csv','semantic_mention_by_condition_band.csv','analysis_semantic.md','manifest.json']:
        publish_immutable(GRADE_DEST+'/'+filename,(grade_dir/filename).read_bytes(),'Save separately versioned semantic grade '+filename)
    print('API grade results:', 'https://github.com/'+REPO+'/tree/main/'+GRADE_DEST)

## 10. 実行後に確認すること

1. コア完了マニフェストの `status=COMPLETED_CORE_PILOT_SEMANTIC_GRADING_SEPARATE` と **17個のチェックポイント**が揃っていることを確認。
2. Cue別言及率とProbe条件差を見て、Xへの明示言及とX/Y識別値の変化が一致するか、**一致しないか**を調べる。
3. 第24層のLogit Lensと通常logitsの一致テストが通っていることを確認。
4. API判定を行った場合は `semantic_grading/.../` へ別保存される。人間による独立ラベル監査は別途必要。
5. パイロットの観察結果を見て刺激を事後的に選別しない。変更は新しいrun ID、明示的な探索分析として扱う。

**未実行の段階では、グラフや出力を予想結果として扱わない。** このノートブックをColabで実行してから、GitHubの実測結果を確認する。